# dataclass 使用

`dataclass` 是 Python 标准库 `dataclasses` 提供的语法糖，用类型注解声明一个数据类，自动生成 `__init__`、`__repr__` 等方法。LangChain 的 `with_structured_output` 也支持用 `dataclass` 作为 schema，底层会解析 `__init__` 的签名把字段转换成 JSON Schema，最终返回一个普通 `dict`。

In [1]:
import json
import os
from dataclasses import dataclass, field
from typing import Annotated, Literal, Optional

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

# 与其他示例保持一致：关闭思考模式，避免结构化输出时 tool_choice 不被支持
model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


## 基础用法

In [2]:
## 基础用法：用 dataclass 描述结构化输出

@dataclass
class Person:
    """人物信息"""
    name: str          # 姓名
    age: int           # 年龄
    occupation: str    # 职位

model_person = model.with_structured_output(schema=Person)
response = model_person.invoke("小许是一个28岁的Java开发工程师")

print(response)
print("返回类型：", type(response))   # dataclass 会返回普通 dict，而不是 dataclass 实例


{'name': '小许', 'age': 28, 'occupation': 'Java开发工程师'}
返回类型： <class 'dict'>


## 字段描述 Annotated

In [3]:
## 字段描述：dataclass 没有 Field()，用 Annotated 给字段附加说明
## 注意：dataclass 是通过解析 __init__ 签名转换的，field(metadata={"description": ...}) 不会生效

@dataclass
class Movie:
    """电影信息"""
    title: Annotated[str, "电影的标题"]
    director: Annotated[str, "导演"]
    year: Annotated[int, "上映年份"]
    rating: Annotated[float, "评分"]

model_movie = model.with_structured_output(schema=Movie)
print(model_movie.invoke("帮我找一下环太平洋电影的信息"))


{'title': '环太平洋', 'director': '吉尔莫·德尔·托罗', 'year': 2013, 'rating': 7.0}


## 默认值

In [4]:
## 默认值：没有默认值的字段是必填，有默认值的字段是可选

@dataclass
class Article:
    """文章信息"""
    title: str                                      # 必填
    author: str = "匿名"                            # 默认值
    views: int = 0                                  # 默认值
    tags: list[str] = field(default_factory=list)   # 可变默认值要用 default_factory

model_article = model.with_structured_output(schema=Article)
print(model_article.invoke("文章《LangChain 入门》作者小许"))


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2448: PydanticJsonSchemaWarning: Default value <factory> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2448: PydanticJsonSchemaWarning: Default value <factory> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2448: PydanticJsonSchemaWarning: Default value <factory> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


{'title': 'LangChain 入门', 'author': '小许'}


## 可选字段

In [5]:
## 可选字段：用 Optional / | None 表示，通常配合默认值 None

@dataclass
class Contact:
    """联系方式"""
    name: str
    phone: Optional[str] = None   # 可以为空
    email: str | None = None      # 同上，Python 3.10+ 的简写

model_contact = model.with_structured_output(schema=Contact)
print(model_contact.invoke("小许，电话 13812345678"))


{'name': '小许', 'phone': '13812345678'}


## 字面量 Literal

In [6]:
## 固定取值：用 Literal 限定字段只能取几个值

@dataclass
class Resume:
    """简历信息"""
    name: str
    level: Literal["初级", "中级", "高级", "专家"]   # 只能取其中之一
    city: str

model_resume = model.with_structured_output(schema=Resume)
print(model_resume.invoke("张三是高级Java开发工程师，base 在上海"))


{'name': '张三', 'level': '高级', 'city': '上海'}


## 列表字段

In [7]:
## 列表字段：让模型一次输出多个同类型元素

@dataclass
class MovieList:
    """电影信息"""
    title: str
    actors: list[str]     # 主演列表
    genres: list[str]     # 类型标签列表

model_movie_list = model.with_structured_output(schema=MovieList)
print(model_movie_list.invoke("请介绍电影《流浪地球》的主演和类型"))


{'title': '流浪地球', 'actors': ['吴京', '屈楚萧', '李光洁', '吴孟达', '赵今麦'], 'genres': ['科幻', '灾难', '冒险']}


## 嵌套 dataclass

In [8]:
## 嵌套：一个 dataclass 可以作为另一个 dataclass 的字段类型

@dataclass
class Address:
    """住址信息"""
    city: str       # 城市
    street: str     # 街道

@dataclass
class Employee:
    """员工信息"""
    name: str
    address: Address   # 字段类型是另一个 dataclass

model_employee = model.with_structured_output(schema=Employee)
print(model_employee.invoke("小许住在北京市朝阳区望京街道"))


{'name': '小许', 'address': {'city': '北京市', 'street': '朝阳区望京街道'}}


## 查看转换后的 JSON Schema

In [9]:
## dataclass 最终会被转换成 JSON Schema 交给模型，打印出来便于排查问题
from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(Employee)["function"], ensure_ascii=False, indent=2))


{
  "name": "Employee",
  "description": "员工信息",
  "parameters": {
    "properties": {
      "name": {
        "type": "string"
      },
      "address": {
        "properties": {
          "city": {
            "type": "string"
          },
          "street": {
            "type": "string"
          }
        },
        "required": [
          "city",
          "street"
        ],
        "type": "object"
      }
    },
    "required": [
      "name",
      "address"
    ],
    "type": "object"
  }
}


## 注意事项

1. **返回的是 `dict`**：dataclass 不是 Pydantic 模型，`with_structured_output` 不会做运行时校验。
2. **字段描述用 `Annotated`**：dataclass 通过解析 `__init__` 签名生成 schema，`field(metadata={"description": ...})` 不会写入描述。
3. **可选字段用默认值表达**：有默认值的字段才不会出现在 `required` 里。
4. **不能直接用于 `@tool(args_schema=...)`**：工具的参数 schema 目前只接受 Pydantic 模型或 JSON Schema `dict`，传 dataclass 会报错；给工具用请改用 Pydantic 或 `TypedDict`。

## 四种 schema 定义方式对比

| 方式 | 运行时校验 | 字段描述 | 返回类型 |
| --- | --- | --- | --- |
| dataclass | 无 | `Annotated` | `dict` |
| JSON Schema `dict` | 无 | schema 里的 `description` | `dict` |
| `TypedDict` | 无 | `Annotated` | `dict` |
| Pydantic | 有 | `Field(description=...)` | 模型实例 |

它们最终都会转换成 JSON Schema 交给模型，选择取决于是否需要类型提示与运行时校验。